# RT-DETR building detection: download, fine-tune and infer

One notebook for the whole pipeline. It runs on **Google Colab**, **Kaggle** or your own machine, and everything runs through **uv**.

| Step | What it does | Command behind it |
|---|---|---|
| 1 | Set up the folder (clone from GitHub on Colab/Kaggle) and install with uv | `uv sync` |
| 2 | Download an RT-DETR checkpoint (any variant from the hub) | `download_model.py` |
| 3 | Download the COGs listed in `list_of_ids.txt` | `download_images.py --list list_of_ids.txt` |
| 4 | Resample and tile the scenes (optional, for labelling) | `tile_cogs.py` |
| 5 | Fine-tune RT-DETR with PyTorch Lightning, configured by Hydra | `train.py` |
| 6 | Detect buildings over the COGs with rasterio windows and write GeoJSON | `infer.py` |

**Settings you can change are in the cells below.** Everything else is controlled by the YAML files in `configs/` and can be overridden on the command line, for example `train.freeze_backbone=true`.

## 1. Set up

On Colab or Kaggle this clones the repository. Set `REPO_URL` to your GitHub repository. On your own machine it just uses the folder you have.

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # Colab sets an inline backend that the uv venv cannot load
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-user>/<your-repo>.git"   # set this to your repository
REPO_DIR = "webinar-DL"

if Path("pyproject.toml").exists() and Path("src/model.py").exists():
    pass                                                  # already inside colab_training/
elif Path("colab_training/pyproject.toml").exists():
    os.chdir("colab_training")
else:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(Path(REPO_DIR) / "colab_training")

print("working folder:", Path.cwd())

In [ ]:
!pip install -q uv
!nvidia-smi -L || echo "no GPU found: training will run on CPU (much slower)"
!uv --version

In [ ]:
!uv sync

## 2. Download an RT-DETR checkpoint

`PekingU/rtdetr_r50vd` is trained on COCO. Any RT-DETR on the Hugging Face hub works, for example `PekingU/rtdetr_r101vd`. The download is saved to `models/` so it can be reused offline.

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # Colab sets an inline backend that the uv venv cannot load

HUB_ID = "PekingU/rtdetr_r50vd"
MODEL_DIR = "models/rtdetr_r50"

!uv run python download_model.py --hub {HUB_ID} --out {MODEL_DIR}

## 3. Download the COGs

The item IDs are in `list_of_ids.txt`. The COGs go to `cog/` and a `manifest.csv` records the licence and producer of each one.

In [ ]:
!uv run python download_images.py --list list_of_ids.txt --out cog

import pandas as pd
manifest = pd.read_csv("cog/manifest.csv")
manifest[["item_id", "status", "size_mb", "gsd_m", "producer", "license"]]

## 4. Resample and tile (for labelling)

Training and inference must use the **same ground resolution**. The project uses 10 cm/px. The tiles from this step are the images to annotate in labelme. Annotate them, then put the `.tif` tiles and their `.json` files in `data/labelme/images/`.

Set `DO_TILING = False` if you already have labelled tiles.

In [ ]:
DO_TILING = True
GSD = 0.10          # m/px used for training and inference
TILE = 640

if DO_TILING:
    !uv run python tile_cogs.py --input cog --gsd {GSD} --tile {TILE} --overlap 128 --output data/tiles --overwrite


## 5. Check the labelled tiles

Training uses tiles that have a labelme `.json` next to them. The labels map to classes in `configs/data/buildings.yaml`.

In [ ]:
from src.labels import read_labelme_samples
import yaml
from pathlib import Path

data_cfg = yaml.safe_load(Path("configs/data/buildings.yaml").read_text())
samples = read_labelme_samples(data_cfg["labels_dir"], data_cfg["label_map"], data_cfg["min_area_ratio"],
                               data_cfg["include_unlabelled"])
n_boxes = sum(len(s["boxes"]) for s in samples)
print(f"{len(samples)} labelled tiles from {len({s['scene'] for s in samples})} scenes, {n_boxes} boxes")
if not samples:
    raise SystemExit("No labelled tiles found: add .tif tiles and their labelme .json files to " + data_cfg["labels_dir"])

## 6. Fine-tune

Training settings live in `configs/train/finetune.yaml` (batch size, learning rate, epochs, freezing). Change them here with overrides, for example:

- `train.freeze_backbone=true` keeps the ResNet backbone fixed
- `train.freeze_decoder_layers=2` freezes the first two decoder layers
- `train.batch_size=2 train.accumulate_grad_batches=4` for a small GPU
- `train.epochs=10` for a quick run

The run writes its checkpoints, the split and the metrics to `outputs/<timestamp>/`. The test set is evaluated at the end, and the best checkpoint is exported to `hf_model/`.

In [ ]:
EPOCHS = 10
OVERRIDES = [
    f"train.epochs={EPOCHS}",
    "train.batch_size=4",
    f"model.checkpoint={MODEL_DIR}",
    # "train.freeze_backbone=true",
    # "train.freeze_decoder_layers=2",
]
!uv run python train.py {" ".join(OVERRIDES)}

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

runs = sorted(Path("outputs").glob("*/logs/version_*/metrics.csv"))
if not runs:
    raise SystemExit("No training metrics found yet: run the training cell first.")
metrics = pd.read_csv(runs[-1])
print("metrics:", runs[-1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for col in [c for c in metrics.columns if c.endswith("loss") and metrics[c].notna().any()]:
    s = metrics[["epoch", col]].dropna()
    axes[0].plot(s["epoch"], s[col], marker="o", label=col)
axes[0].set_title("loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
for col in [c for c in metrics.columns if "map" in c and metrics[c].notna().any()]:
    s = metrics[["epoch", col]].dropna()
    axes[1].plot(s["epoch"], s[col], marker="o", label=col)
axes[1].set_title("COCO mAP")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Infer over the COGs

`infer.py` reads 640 × 640 windows from each COG with rasterio, predicts on them through Lightning, merges the overlapping detections and writes one GeoJSON per class (lon/lat, WGS84) to `outputs/geojson/`.

The model is the one exported by the last training run, unless you set `MODEL_CHECKPOINT` to another folder or checkpoint.

In [ ]:
exported = sorted(Path("outputs").glob("*/hf_model"))
MODEL_CHECKPOINT = str(exported[-1]) if exported else MODEL_DIR
print("model:", MODEL_CHECKPOINT)

COG_FILE = "cog/68a4a953e50784296730bf0f.tif"       # one scene; set to None to run every COG in cog/
SCORE_THRESHOLD = 0.4
infer_overrides = [
    f"model.checkpoint={MODEL_CHECKPOINT}",
    f"inference.score_threshold={SCORE_THRESHOLD}",
    f"inference.gsd={GSD}",
]
if COG_FILE:
    infer_overrides.append(f"inference.cog={COG_FILE}")
!uv run python infer.py {" ".join(infer_overrides)}

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

files = sorted(Path("outputs/geojson").rglob("*.geojson"))
print(f"{len(files)} GeoJSON files")
fig, ax = plt.subplots(figsize=(9, 9))
for f in files:
    fc = json.loads(f.read_text())
    for feat in fc["features"]:
        ring = feat["geometry"]["coordinates"][0]
        xs, ys = zip(*ring)
        ax.fill(xs, ys, alpha=0.35, edgecolor="k", linewidth=0.3)
    print(f"  {f.name}: {len(fc['features'])} features")
ax.set_aspect("equal")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Predicted buildings (lon/lat)")
plt.show()

## 8. FiftyOne dataset (5 cm tiles)

`src/fo_dataset.py` tiles every COG in `cog/` at **5 cm/px** into **640 × 640** tiles (128 px overlap) and adds them to a FiftyOne dataset named `building_tiles_5cm`.

The labelme annotations in `data/labelme/images/` become the field `ground_truth` (boxes). Only buildings are kept: `building` and `building-ext` are merged into `building`, `lote` (parcels) is dropped. A tile has labels only when it lies inside the footprint of an annotated tile (`annotated=True`).

The tiles go to `data/tiles_5cm/` (ignored by git). The resampled COGs go to `cog/resampled_5cm/`, so the 10 cm files from step 4 are not reused.

In [ ]:
from src.fo_dataset import build_dataset, dataset_summary

# True: tile the COGs again (first run, or after changing the tiling). False: load the dataset that already exists.
RETILE = True
dataset = build_dataset(gsd=0.05, tile=640, overlap=128, retile=RETILE, overwrite=RETILE)
dataset_summary(dataset)

## 9. Zero-shot test (no fine-tuning)

Tag a few annotated tiles as `test`, then run pretrained models on them and score the boxes against `ground_truth` (IoU 0.5, mAP over IoU 0.5:0.95).

- **rtdetr**: COCO RT-DETR has no building class, so every COCO detection counts as a building. This is a baseline.
- **owlv2**: open-vocabulary detector, prompted with text queries.
- **sam2**: automatic masks, class-agnostic, kept by score and footprint area.

In [ ]:
from src.fo_dataset import tag_test_samples

TEST_TILES = 10                                   # how many annotated tiles to tag as "test"
test_view = tag_test_samples(dataset, n=TEST_TILES, seed=51, tag="test")
# import fiftyone as fo; session = fo.launch_app(test_view)   # optional: browse the test tiles and their labels

In [ ]:
import torch
from src.zeroshot import pick_device, load_detector, run_field, run_detector, evaluate_run, compare_runs

DEVICE = pick_device()
print("device:", DEVICE)

# Edit this list to change the models or the weights.
#   kind:    rtdetr | owlv2 | sam2
#   weights: a Hugging Face hub id, or a local folder (e.g. outputs/<run>/hf_model after fine-tuning)
ZERO_SHOT_RUNS = [
    dict(kind="rtdetr", weights="PekingU/rtdetr_r50vd", score_threshold=0.3),
    dict(kind="owlv2", weights="google/owlv2-base-patch16-ensemble", score_threshold=0.2, queries=["a building"]),
    dict(kind="sam2", weights="facebook/sam2.1-hiera-large", score_threshold=0.85, min_area_m2=10, max_area_m2=1500),
]
ACTIVE = [0, 1, 2]                                # indices of ZERO_SHOT_RUNS to run now, e.g. [1] for OWLv2 only

In [ ]:
test_view = dataset.match_tags("test")
summary = {}
for i in ACTIVE:
    options = dict(ZERO_SHOT_RUNS[i])
    kind, weights = options.pop("kind"), options.pop("weights")
    field = run_field(kind, weights)
    print(f"\n== {kind}: {weights} -> field '{field}'")
    detector = load_detector(kind, weights, DEVICE, **options)
    run_detector(test_view, detector, field)
    summary[field] = evaluate_run(test_view, field)
    del detector                                  # free the GPU before the next model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results_table = compare_runs(summary)

## 10. Next steps

- Label more tiles in labelme and re-run from step 5. The more scenes there are, the better the split by scene works.
- Compare variants: change `MODEL_DIR` and `HUB_ID` in step 2 (for example `PekingU/rtdetr_r101vd`).
- Lower `SCORE_THRESHOLD` for more recall, raise it for more precision.
- The GeoJSON files open directly in QGIS.